In [1]:
import pandas as pd

# Load data
train = pd.read_csv('/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv')
test = pd.read_csv('/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv')
dictionary = pd.read_csv('/kaggle/input/competitions/deep-past-initiative-machine-translation/eBL_Dictionary.csv')
sample_sub = pd.read_csv('/kaggle/input/competitions/deep-past-initiative-machine-translation/sample_submission.csv')

print(f"Train size : {len(train)}")
print(f"Test size  : {len(test)}")
print(f"Dict size  : {len(dictionary)}")
print(f"\nTrain columns : {train.columns.tolist()}")
print(f"Test columns  : {test.columns.tolist()}")
print(f"\n{train.head(3).to_string()}")


Train size : 1561
Test size  : 4
Dict size  : 19215

Train columns : ['oare_id', 'transliteration', 'translation']
Test columns  : ['id', 'text_id', 'line_start', 'line_end', 'transliteration']

                                oare_id                                                                                                                                                                                                                                                                                                                                                                        transliteration                                                                                                                                                                                                                                                                                                                                                                                              translati

In [2]:
train["src_len"] = train["transliteration"].str.split().str.len()
train["tgt_len"] = train["translation"].str.split().str.len()

print("Source (transliteration) length stats:")
print(train["src_len"].describe().to_string())
print("\nTarget (translation) length stats:")
print(train["tgt_len"].describe().to_string())

print(f"\nMax source tokens : {train['src_len'].max()}")
print(f"Max target tokens : {train['tgt_len'].max()}")
print(f"\n95th percentile source: {train['src_len'].quantile(0.95):.0f}")
print(f"95th percentile target: {train['tgt_len'].quantile(0.95):.0f}")

print("\n\nTest samples:")
print(test.to_string())


Source (transliteration) length stats:
count    1561.000000
mean       56.157591
std        35.511692
min         3.000000
25%        27.000000
50%        49.000000
75%        83.000000
max       150.000000

Target (translation) length stats:
count    1561.000000
mean       89.987828
std        84.963638
min         1.000000
25%        30.000000
50%        68.000000
75%       124.000000
max       739.000000

Max source tokens : 150
Max target tokens : 739

95th percentile source: 119
95th percentile target: 247


Test samples:
   id   text_id  line_start  line_end                                                                                                                                                                                                                                                              transliteration
0   0  332fda50           1         7                                                                                                                           

In [3]:
# ── Dictionary lookup (Akkadian word → English definition) 
print(f"Dictionary columns: {dictionary.columns.tolist()}")
print(dictionary.head(5).to_string())

# Build lookup dict
akk_to_eng = dict(zip(
    dictionary["word"].str.lower().str.strip(),
    dictionary["definition"].fillna("")
))
print(f"\nDictionary entries: {len(akk_to_eng)}")
print("Sample entries:")
for k, v in list(akk_to_eng.items())[:5]:
    print(f"  {k!r:30s} → {v[:60]!r}")


Dictionary columns: ['word', 'definition', 'derived_from']
      word                                                           definition derived_from
0     -a I                                             "my" (1 sg. pron. suff.)     cf. -ī I
1    -am I  "to me" (1 sg. dat. suff.) 2. vent. affix (cf. GAG p.251); cf. -nim          NaN
2  -atti I                                       (adv. endings) (cf. GAG §113l)          NaN
3    -iš I       "to; like" term.-adv. suffix on subst.s and adj.s; cf. GAG §67          NaN
4    -ka I                         "you" (2 m. sg. acc. suff.) (cf. GAG p.258b)          NaN

Dictionary entries: 19079
Sample entries:
  '-a i'                         → '"my" (1 sg. pron. suff.)'
  '-am i'                        → '"to me" (1 sg. dat. suff.) 2. vent. affix (cf. GAG p.251); c'
  '-atti i'                      → '(adv. endings) (cf. GAG §113l)'
  '-iš i'                        → '"to; like" term.-adv. suffix on subst.s and adj.s; cf. GAG §'
  '-ka i'     

In [4]:
! pip install evaluate sacrebleu sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.7 MB/s eta 0:00:00


In [5]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from datasets import Dataset
import evaluate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Torch version: {torch.__version__}")


Device: cuda
Torch version: 2.9.0+cu126


In [6]:
#Model & Tokenizer
# mt5-small is ideal for low-resource fine-tuning
MODEL_NAME = "google/mt5-small"
PREFIX = "translate Akkadian to English: "
MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)

print(f"Model parameters: {model.num_parameters():,}")


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model parameters: 556,291,456


In [7]:
from sklearn.model_selection import train_test_split

# Split off 10% validation (keeping all 4 test rows separate)
tr, val = train_test_split(train, test_size=0.10, random_state=42)
print(f"Train: {len(tr)} | Val: {len(val)}")

def make_hf_dataset(df):
    return Dataset.from_dict({
        "src": (PREFIX + df["transliteration"]).tolist(),
        "tgt": df["translation"].tolist(),
    })

hf_train = make_hf_dataset(tr)
hf_val   = make_hf_dataset(val)

def tokenize(batch):
    # Tokenize source
    model_inputs = tokenizer(
        batch["src"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,
    )
    # Tokenize target separately with its own max_length
    labels = tokenizer(
        text_target=batch["tgt"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

hf_train = hf_train.map(tokenize, batched=True, remove_columns=["src", "tgt"])
hf_val   = hf_val.map(tokenize,   batched=True, remove_columns=["src", "tgt"])

print("Tokenization complete.")
print(f"Example input_ids length : {len(hf_train[0]['input_ids'])}")
print(f"Example labels length    : {len(hf_train[0]['labels'])}")


Train: 1404 | Val: 157


Map:   0%|          | 0/1404 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

Tokenization complete.
Example input_ids length : 116
Example labels length    : 76


In [8]:
# ── Metrics (chrF + BLEU) ────────────────────────────────────────────────────
import numpy as np

chrf_metric = evaluate.load("chrf")
bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # preds may be a tuple when decoder_input_ids are returned
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace out-of-range / negative values with pad token id
    preds  = np.where(preds  >= 0, preds,  tokenizer.pad_token_id).astype(np.int32)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id).astype(np.int32)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]

    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)
    bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {

        "chrF": round(chrf["score"], 4),    
        "bleu": round(bleu["score"], 4),}

In [9]:
# ── Training Arguments & Trainer ─────────────────────────────────────────────
OUTPUT_DIR = r"/kaggle/working/"

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    # ── Training schedule ────────────────────────────────────────────────────
    num_train_epochs=30,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=3e-4,
    max_grad_norm=1.0,
    # ── Precision ────────────────────────────────────────────────────────────
    fp16=False,
    bf16=torch.cuda.is_available(),
    # ── Generation settings for eval ─────────────────────────────────────────
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    # ── Evaluation & saving ───────────────────────────────────────────────────
    eval_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="chrF",
    greater_is_better=True,
    # ── Logging ──────────────────────────────────────────────────────────────
    logging_steps=10,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print("Trainer ready. Starting training…")
trainer.train()
print("Training complete!")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer ready. Starting training…


Epoch,Training Loss,Validation Loss,Chrf,Bleu
1,7.373789,5.418614,1.841700,0.114600
2,3.278251,2.901821,10.501300,1.054900
3,2.926736,2.541684,12.584200,2.324200
4,2.606459,2.312988,15.340900,3.286400
5,2.363445,2.160273,19.516400,4.264300
6,2.187000,2.084758,22.044000,5.763400
7,2.193486,2.017602,24.237900,7.200500
8,2.184071,1.974263,27.144100,8.473300
9,1.956348,1.932268,27.047700,8.816900
10,1.920828,1.912952,28.623200,11.149800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [10]:
#  Inference on test set 
model.eval()

def translate(texts, batch_size=4):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = [PREFIX + t for t in texts[i:i+batch_size]]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(DEVICE)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_TARGET_LEN,
                num_beams=4,
                early_stopping=True,
            )
        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        results.extend(decoded)
    return results

test_translations = translate(test["transliteration"].tolist())

for src, tgt in zip(test["transliteration"].tolist(), test_translations):
    print(f"SRC: {src[:80]}...")
    print(f"TGT: {tgt}")
    print()


SRC: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa...
TGT: From the Kanesh colony to Aššur's dagger, our messenger, and our messenger: I am staying in the City. I am staying in the City.

SRC: i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-nim ma-ma-an KÙ.AN i-aa-ú-mu-ni i-n...
TGT: I am staying in the City. My dear fathers, when they arrive from the City they must not raise claim against us. The Kanesh colony gave us for these proceedings and we gave our testimony before Aššur's dagger.

SRC: ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É.GAL-lim i-dí-in l...
TGT: From the merchant to the merchant: Why is it that when the colony comes up to the colony when the colony goes to the colony? Why is it that the colony refuses to come to the colony? Why is it that the colony refuses to go to the colony? Why is it that the colony refuses to go to the colony? Why is it that the colony refuses to go to the colony? Why is it that the colony refuse

In [12]:
# Create submission CSV
submission = pd.DataFrame({
    "id":          test["id"].tolist(),
    "translation": test_translations,
})

print(submission.to_string())

SUBMIT_PATH = "/kaggle/working/submission.csv"
submission.to_csv(SUBMIT_PATH, index=False)
print(f"\nSubmission saved to: {SUBMIT_PATH}")


   id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             translation
0   0                                                                                                                                                                                                     